# 09 — Неограниченный LIMIT (нет пагинации)

> **`vuln_class`:** `NO_PAGINATION` · **Риск:** 4/10 · **CWE-770**

Запрос без `LIMIT` на большой таблице тащит всё в память приложения. На пороге одобрения (`RISK_THRESHOLD = 4.0`) — сам по себе пропустится, но в комбинации с `SELECT *` (5) или `DIRECT_SENSITIVE` (6) превышает.


## 🧒 Аналогия для ребёнка

Ты приходишь в библиотеку и говоришь библиотекарю:
«**дай мне все книги** про динозавров».

- **Плохо:** библиотекарь катит тебе **5 тележек** с 500 книгами.
  Ты не унесёшь, бросишь половину, и всем плохо.
- **Хорошо:** «**дай мне 10 книг** про динозавров, отсортированных
  по году издания». Берёшь 10, читаешь, возвращаешься за следующими.

Это **пагинация**. В SQL — `LIMIT 10`. Без LIMIT БД отдаст
ВСЁ — миллион строк, гигабайты трафика, OOM в приложении.


## 1. Setup — большая таблица


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


def setup_orders():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, user_id INTEGER, amount REAL, ts TEXT)")
    # 10000 строк — для демо «большой» таблицы
    orders = [(i, i % 100, i * 1.5, f"2026-01-{(i % 28) + 1:02d}") for i in range(1, 10001)]
    conn.executemany("INSERT INTO orders (id, user_id, amount, ts) VALUES (?, ?, ?, ?)", orders)
    conn.commit()
    return conn


conn = setup_orders()
n = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
print(f"В таблице orders: {n} строк")


## 2. Уязвимый запрос — нет LIMIT


In [ ]:
##
# @brief УЯЗВИМАЯ функция: тянет все заказы.
def fetch_all_orders_BAD(conn):
    sql = "SELECT id, user_id, amount, ts FROM orders ORDER BY ts DESC"
    print(f"  SQL: {sql}")
    t0 = time.time()
    rows = conn.execute(sql).fetchall()
    dt = time.time() - t0
    print(f"  Получено {len(rows)} строк за {dt*1000:.1f} мс")
    return rows


section("Уязвимый вызов — тянет всё")
rows = fetch_all_orders_BAD(conn)


## 3. Маскированная версия — `LIMIT 1_000_000`

Иногда разработчик «защищается» гигантским LIMIT — это та же
проблема, просто менее очевидная.


In [ ]:
##
# @brief Маскированная версия: LIMIT есть, но абсурдный.
def fetch_orders_BAD_huge_limit(conn):
    sql = "SELECT id, user_id, amount, ts FROM orders ORDER BY ts DESC LIMIT 1000000"
    rows = conn.execute(sql).fetchall()
    print(f"  Получено {len(rows)} строк (LIMIT 1000000)")


fetch_orders_BAD_huge_limit(conn)


## 4. Аудитор Phase 1 — `R004-no-limit`


In [ ]:
##
# @brief Phase 1 R004 — детект SELECT без LIMIT или с гигантским LIMIT.
def audit_R004_no_limit(sql_text):
    findings = []
    is_select = bool(re.match(r"\s*SELECT\b", sql_text, re.IGNORECASE))
    if not is_select:
        return findings
    # Игнорируем агрегаты (мало строк)
    if re.search(r"\b(COUNT|SUM|AVG|MIN|MAX|GROUP\s+BY)\b", sql_text, re.IGNORECASE):
        return findings

    limit_m = re.search(r"\bLIMIT\s+(\d+)", sql_text, re.IGNORECASE)
    if limit_m is None:
        findings.append({
            "rule_id":       "R004-no-limit",
            "vuln_class":    "NO_PAGINATION",
            "severity":      "low", "risk_score": 4,
            "message":       "SELECT без LIMIT — потенциальный DoS",
            "evidence_refs": ["CWE-770"],
        })
    else:
        n = int(limit_m.group(1))
        if n > 10_000:
            findings.append({
                "rule_id":       "R004-no-limit",
                "vuln_class":    "NO_PAGINATION",
                "severity":      "low", "risk_score": 3,
                "message":       f"LIMIT {n} — избыточен, по сути нет пагинации",
                "evidence_refs": ["CWE-770"],
            })
    return findings


section("Аудитор по разным SQL")
for sql in [
    "SELECT id, user_id FROM orders ORDER BY ts DESC",
    "SELECT id, user_id FROM orders ORDER BY ts DESC LIMIT 1000000",
    "SELECT id, user_id FROM orders ORDER BY ts DESC LIMIT 50",
    "SELECT COUNT(*) FROM orders",  # это норм — агрегат
]:
    print(f"\n SQL: {sql}")
    fs = audit_R004_no_limit(sql)
    if fs:
        for f in fs:
            print_finding(f)
    else:
        print("  ✅ ok")


## 5. Безопасная версия — keyset pagination


In [ ]:
##
# @brief Безопасная функция: keyset pagination.
# @details
#   Вместо OFFSET (медленно на больших таблицах) — фильтр по id курсора.
#   Каждый запрос — стабильно O(page_size).
def fetch_orders_GOOD(conn, last_seen_id: int = None, page_size: int = 50):
    if last_seen_id is None:
        sql = "SELECT id, user_id, amount, ts FROM orders ORDER BY id DESC LIMIT ?"
        return conn.execute(sql, (page_size,)).fetchall()
    sql = "SELECT id, user_id, amount, ts FROM orders WHERE id < ? ORDER BY id DESC LIMIT ?"
    return conn.execute(sql, (last_seen_id, page_size)).fetchall()


section("Первая страница (50 строк)")
page1 = fetch_orders_GOOD(conn, page_size=5)  # для демо берём 5
for r in page1:
    print(f"  {r}")


section("Следующая страница — last_seen_id = id последнего из page1")
last_id = page1[-1][0]
page2 = fetch_orders_GOOD(conn, last_seen_id=last_id, page_size=5)
for r in page2:
    print(f"  {r}")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/09-no-pagination/README.md](../../problems/vulnerabilities/09-no-pagination/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/09-no-pagination/solutions.md](../../problems/vulnerabilities/09-no-pagination/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
